# QLoRA fine-tuning for code refinement — Colab / cloud GPU runner

Runs the **exact same pipeline** as the repo, on a GPU. Nothing here is a
notebook-only reimplementation: every cell shells out to the same `coderefine`
CLI you run locally, so results are reproducible outside Colab.

**Runtime → Change runtime type → T4 GPU** (or better) before running.

| GPU | VRAM | Mistral-7B QLoRA, 2k examples, 3 epochs |
|---|---|---|
| T4 (free tier) | 16 GB | ~50–70 min |
| L4 | 24 GB | ~25–35 min |
| A100 | 40 GB | ~10–15 min |

In [10]:
# 0 — check the GPU we were given
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.82.07


## 1 — Get the code and the data

Two options. **A** clones your repo (put your URL in). **B** uploads a zip of
the project folder — use it if the repo is private.

The raw `Code_Refinement/*.jsonl` dumps are ~6 GB, which is too large to upload
comfortably. Either mount Drive with them, or — much faster — copy up only the
**already-curated** `data/` directory (a few MB), which is all training needs.

In [11]:
# 1A — clone the repo (curated data is committed, so no raw-dump download needed)REPO_URL = "https://github.com/hamnaraeel/lora-code-refinement.git"!rm -rf lora-code-refinement!git clone -q $REPO_URL%cd lora-code-refinement!ls -la data/processed data/benchmark

Skipping clone — using direct zip upload in the next cell.


In [12]:
# 1B — not needed this run: 1A already cloned everything, curated data included.print("Skipping — data/processed and data/benchmark came with the git clone above.")

Select coderefine_project.zip when prompted...


KeyboardInterrupt: 

In [ ]:
# 1C — not needed this run (no raw dumps to fetch; curated splits are already in the repo)

## 2 — Install

`bitsandbytes` is what makes 4-bit QLoRA possible and only exists for CUDA,
which is exactly why the local Intel-Mac path in this repo cannot run it.

In [ ]:
# 2 — Install## Pinned to the exact combo verified end-to-end on this project (real training# run completed, 144 tests passing): transformers 4.46.3 / peft 0.13.2 /# trl 0.11.4 / accelerate 1.2.1. This matters more than it looks: current# Colab images ship a much newer TRL that replaced the completion-only-loss# collator with a chat-template-marker mechanism TRL only auto-patches for a# small allowlist of models (Mistral is not on it), which fails outright# rather than silently mis-masking anything. Pinning below avoids that entirely# by using the older, thoroughly-tested masking path.!pip install -q "transformers==4.46.3" "peft==0.13.2" "trl==0.11.4" "accelerate==1.2.1" "tokenizers<0.21" \    "bitsandbytes>=0.43.1" "datasets>=2.19" sacrebleu Levenshtein pyyaml pydantic typer rich matplotlib wandb!pip install -q -e . --no-depsimport sysif "/content/lora-code-refinement/src" not in sys.path:    sys.path.insert(0, "/content/lora-code-refinement/src")import torch, transformers, peft, trl, accelerate, bitsandbytesprint("torch", torch.__version__, "| cuda", torch.cuda.is_available())if torch.cuda.is_available():    print("gpu:", torch.cuda.get_device_name(0), "| vram_gb:", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1))print("transformers", transformers.__version__, "| peft", peft.__version__, "| trl", trl.__version__, "| accelerate", accelerate.__version__)print("bitsandbytes", bitsandbytes.__version__)from trl import DataCollatorForCompletionOnlyLM  # must succeed — confirms the legacy masking path is activeprint("legacy masking path: OK")

## 3 — Credentials

* **Hugging Face** — needed for gated bases (Llama 3). Mistral-7B-Instruct-v0.3 is ungated.
* **Weights & Biases** — optional. Without a key the pipeline still logs
  everything to `artifacts/runs/<name>/metrics.jsonl`; it degrades, it does not fail.

In [ ]:
import os, getpass

# from huggingface_hub import login; login()   # only needed for gated models

use_wandb = False  # set True to log to W&B
if use_wandb:
    os.environ["WANDB_API_KEY"] = getpass.getpass("W&B API key: ")
    os.environ["WANDB_PROJECT"] = "lora-code-refinement"

## 4 — Build the dataset

Skip this if you copied a prepared `data/` directory up. It needs the raw
`Code_Refinement/*.jsonl` dumps present.

In [ ]:
import pathlib
if pathlib.Path("Code_Refinement/ref-train.jsonl").exists():
    !coderefine build-data --n-train 2000 --n-valid 250 --n-test 250
    !coderefine build-benchmark
else:
    print("Raw dumps absent — assuming data/ was copied up.")
!ls -la data/processed data/benchmark

## 5 — Train

One command, one config file. Everything that affects the result lives in the
YAML, so this run is reproducible anywhere.

In [ ]:
!coderefine train configs/qlora_mistral7b.yaml

In [ ]:
# Loss curves from the local metric log (works with or without W&B)
import json, pathlib
import matplotlib.pyplot as plt

run = "qlora-mistral7b-r16"
rows = [json.loads(l) for l in (pathlib.Path("artifacts/runs")/run/"metrics.jsonl").read_text().splitlines() if l.strip()]
tr = [(r["_step"], r["loss"]) for r in rows if "loss" in r]
ev = [(r["_step"], r["eval_loss"]) for r in rows if "eval_loss" in r]

fig, ax = plt.subplots(figsize=(8, 4.5))
if tr: ax.plot(*zip(*tr), label="train loss", lw=1.4)
if ev: ax.plot(*zip(*ev), label="validation loss", marker="o", lw=1.6)
if ev:
    bs, bl = min(ev, key=lambda x: x[1])
    ax.axvline(bs, ls="--", c="gray", lw=1)
    ax.annotate(f"best checkpoint\nstep {bs} · {bl:.4f}", (bs, bl),
                textcoords="offset points", xytext=(10, 18), fontsize=9)
ax.set_xlabel("step"); ax.set_ylabel("loss"); ax.set_title(f"{run} — training curves")
ax.legend(); ax.grid(alpha=.25); fig.tight_layout()
fig.savefig("reports/training_curves.png", dpi=150)
plt.show()

## 6 — Hyperparameter sweep

Eight arms varying rank (8/16/32), learning rate (1e-4/2e-4/5e-4), epochs
(1/3/5) and target modules (q,v vs all-linear). On a T4 this is several hours —
run a subset first if you are on the free tier and may be pre-empted.

In [ ]:
# Subset (fast): rank sweep only
!bash scripts/run_sweep.sh configs/sweep/s01_r8_lr2e4_e3.yaml configs/sweep/s02_r16_lr2e4_e3.yaml configs/sweep/s03_r32_lr2e4_e3.yaml

# Everything:
# !bash scripts/run_sweep.sh

## 7 — Evaluate

Base first — it is the denominator of every improvement claim — then the tuned
model on the identical benchmark with the identical prompts and greedy decoding.

In [ ]:
ADAPTER = "artifacts/runs/qlora-mistral7b-r16/adapter"
BASE    = "mistralai/Mistral-7B-Instruct-v0.3"

!coderefine evaluate --split benchmark --base-model $BASE --tag base --load-in-4bit
!coderefine evaluate --split benchmark --base-model $BASE --adapter $ADAPTER --tag tuned --load-in-4bit
!coderefine compare artifacts/eval/base__benchmark.predictions.jsonl \
                    artifacts/eval/tuned__benchmark.predictions.jsonl

In [ ]:
# LLM-as-judge (optional — costs API credits)
# import os, getpass
# os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API key: ")
# !coderefine judge artifacts/eval/base__benchmark.predictions.jsonl \
#                   artifacts/eval/tuned__benchmark.predictions.jsonl --provider anthropic

In [ ]:
# Catastrophic forgetting: general capability, before and after
!coderefine forgetting --base-model $BASE --load-in-4bit
!coderefine forgetting --base-model $BASE --adapter $ADAPTER --load-in-4bit

## 8 — The sacred test split

Only run this once, after the configuration is frozen. Everything above used
validation and the benchmark.

In [ ]:
!coderefine evaluate --split test --final --base-model $BASE --tag base-test  --load-in-4bit
!coderefine evaluate --split test --final --base-model $BASE --adapter $ADAPTER --tag tuned-test --load-in-4bit
!coderefine compare artifacts/eval/base-test__test.predictions.jsonl \
                    artifacts/eval/tuned-test__test.predictions.jsonl \
                    --out-path artifacts/eval/comparison_test.json

## 9 — Report, package, and download

In [ ]:
!coderefine report
!coderefine export $ADAPTER --out-dir artifacts/release --base-model $BASE
from IPython.display import Markdown, display
display(Markdown(open("reports/EXPERIMENT_REPORT.md").read()))

In [ ]:
# Zip the adapter + all evaluation artifacts and pull them down
!zip -qr coderefine_results.zip artifacts/release artifacts/eval artifacts/runs artifacts/forgetting reports data/processed/dataset_card.json data/benchmark/benchmark_card.json
!du -h coderefine_results.zip
from google.colab import files
files.download("coderefine_results.zip")

## 10 — Try the A/B server here

Serves base and fine-tuned from one resident model by toggling the adapter.

In [ ]:
import subprocess, time, requests, json
proc = subprocess.Popen(
    ["coderefine","serve","--base-model",BASE,"--adapter",ADAPTER,"--load-in-4bit","--port","8000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for _ in range(90):
    try:
        if requests.get("http://127.0.0.1:8000/healthz", timeout=2).ok: break
    except Exception: time.sleep(2)
print(requests.get("http://127.0.0.1:8000/healthz").json())

payload = {
 "lang":"py",
 "old_code":"def load_config(path):\n    try:\n        with open(path) as fh:\n            return json.load(fh)\n    except Exception:\n        return None",
 "comment":"This except block swallows the error and returns None, which hides real failures. Just let it propagate.",
 "gold":"def load_config(path):\n    with open(path) as fh:\n        return json.load(fh)",
}
print(json.dumps(requests.post("http://127.0.0.1:8000/ab", json=payload, timeout=300).json(), indent=2))
# proc.terminate()

In [ ]:
import sys, platform, os
print("python:", sys.version)
print("platform:", platform.platform())
try:
    import torch
    print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("gpu:", torch.cuda.get_device_name(0))
        print("vram_gb:", round(torch.cuda.get_device_properties(0).total_memory/1e9,1))
except ImportError:
    print("torch not installed yet")
print("cwd:", os.getcwd())
!nvidia-smi --query-gpu=name,memory.total --format=csv 2>/dev/null || echo "no GPU visible"

python: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
platform: Linux-6.6.122+-x86_64-with-glibc2.35
torch: 2.11.0+cu128 | cuda available: True
gpu: Tesla T4
vram_gb: 15.6
cwd: /content
name, memory.total [MiB]
Tesla T4, 15360 MiB


In [ ]:
import os
os.listdir('/content')

['.config', 'sample_data']

In [ ]:
import os
if os.path.exists('/content/upload.b64'):
    os.remove('/content/upload.b64')
os.makedirs('/content/project', exist_ok=True)
print("ready")

In [ ]:
import os
if os.path.exists('/content/upload.b64'):
    os.remove('/content/upload.b64')
os.makedirs('/content/project', exist_ok=True)
print("ready")

ready


In [ ]:
%cd /content
!rm -rf lora-code-refinement
!git clone -q https://github.com/hamnaraeel/lora-code-refinement.git
%cd lora-code-refinement
!ls -la
print("--- data ---")
!ls -la data/processed data/benchmark

/content
fatal: could not read Username for 'https://github.com': No such device or address
[Errno 2] No such file or directory: 'lora-code-refinement'
/content
total 20
drwxr-xr-x 1 root root 4096 Sep  4 13:56 .
drwxr-xr-x 1 root root 4096 Sep  4 13:35 ..
drwxr-xr-x 4 root root 4096 Aug 24 13:27 .config
drwxr-xr-x 2 root root 4096 Sep  4 13:42 project
drwxr-xr-x 1 root root 4096 Aug 24 13:28 sample_data
--- data ---
ls: cannot access 'data/processed': No such file or directory
ls: cannot access 'data/benchmark': No such file or directory


In [ ]:
%cd /content
!rm -rf lora-code-refinement
!git clone -q https://github.com/hamnaraeel/lora-code-refinement.git
%cd lora-code-refinement
!ls -la
print("--- data ---")
!ls -la data/processed data/benchmark

In [13]:
print("kernel responsive")
import os
print("cwd:", os.getcwd())

kernel responsive
cwd: /content


In [14]:
%cd /content
!rm -rf lora-code-refinement
!git clone -q https://github.com/hamnaraeel/lora-code-refinement.git
%cd lora-code-refinement
!ls -la
print("--- data ---")
!ls -la data/processed data/benchmark

/content
/content/lora-code-refinement
total 76
drwxr-xr-x 10 root root  4096 Sep  4 14:05 .
drwxr-xr-x  1 root root  4096 Sep  4 14:05 ..
drwxr-xr-x  3 root root  4096 Sep  4 14:05 configs
drwxr-xr-x  4 root root  4096 Sep  4 14:05 data
drwxr-xr-x  2 root root  4096 Sep  4 14:05 docker
-rw-r--r--  1 root root   891 Sep  4 14:05 .env.example
drwxr-xr-x  8 root root  4096 Sep  4 14:05 .git
-rw-r--r--  1 root root   923 Sep  4 14:05 .gitignore
-rw-r--r--  1 root root  2963 Sep  4 14:05 Makefile
drwxr-xr-x  2 root root  4096 Sep  4 14:05 notebooks
-rw-r--r--  1 root root   632 Sep  4 14:05 pyproject.toml
-rw-r--r--  1 root root 12202 Sep  4 14:05 README.md
-rw-r--r--  1 root root   807 Sep  4 14:05 requirements-gpu.txt
-rw-r--r--  1 root root  1220 Sep  4 14:05 requirements.txt
drwxr-xr-x  2 root root  4096 Sep  4 14:05 scripts
drwxr-xr-x  3 root root  4096 Sep  4 14:05 src
drwxr-xr-x  2 root root  4096 Sep  4 14:05 tests
--- data ---
data/benchmark:
total 64
drwxr-xr-x 2 root root  4096 

In [ ]:
%cd /content/lora-code-refinement
!pip install -q -U "transformers>=4.44" "peft>=0.12" "trl>=0.9.6" "accelerate>=0.33" "datasets>=2.19" "bitsandbytes>=0.43.1" sacrebleu Levenshtein pyyaml pydantic typer rich matplotlib 2>&1 | tail -15
!pip install -q -e . --no-deps 2>&1 | tail -5
import torch, transformers, peft, trl, bitsandbytes
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
print("transformers", transformers.__version__, "| peft", peft.__version__, "| trl", trl.__version__)
print("bitsandbytes", bitsandbytes.__version__)

In [15]:
%cd /content/lora-code-refinement
!pip install -q -U "transformers>=4.44" "peft>=0.12" "trl>=0.9.6" "accelerate>=0.33" "datasets>=2.19" "bitsandbytes>=0.43.1" sacrebleu Levenshtein pyyaml pydantic typer rich matplotlib 2>&1 | tail -15
!pip install -q -e . --no-deps 2>&1 | tail -5
import torch, transformers, peft, trl, bitsandbytes
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
print("transformers", transformers.__version__, "| peft", peft.__version__, "| trl", trl.__version__)
print("bitsandbytes", bitsandbytes.__version__)

/content/lora-code-refinement
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.7/158.7 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.6/472.6 kB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 99.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.1/123.1 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.7/310.7 kB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 135.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 117.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 13.9 MB/s eta 0:00:00
ERROR: pip